In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [7]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("dohala_chalasani_app") \
    .master("local[*]") \
    .config("spark.driver.host", "localhost") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

data = [("dohala_chalasani",)]
df = spark.createDataFrame(data, ["Name"])

df.show()

+----------------+
|            Name|
+----------------+
|dohala_chalasani|
+----------------+



In [12]:
!pip install requests

In [25]:
import requests

base_url = "https://data.bts.gov/resource/w96p-f2qv.csv"

limit = 50000
total_rows = 5000000
offset = 0

with open("bts_data.csv", "w", encoding="utf-8") as f:
    while offset < total_rows:
        print(f"Downloading rows {offset} to {offset + limit}")

        params = {
            "$limit": limit,
            "$offset": offset
        }

        response = requests.get(base_url, params=params)
        response.raise_for_status()

        text = response.text

        if offset > 0:
            text = "\n".join(text.split("\n")[1:])

        f.write(text)
        offset += limit

print("Download Completed ")

Download Completed 


In [26]:
!ls -lh bts_data.csv

-rw-rw-r-- 1 dohala_chalasani dohala_chalasani 1.1G Mar  2 20:06 bts_data.csv


In [27]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("bts_data.csv")

In [28]:
df.printSchema()

root
 |-- level: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- state_fips: integer (nullable = true)
 |-- state_code: string (nullable = true)
 |-- county_fips: integer (nullable = true)
 |-- county: string (nullable = true)
 |-- pop_stay_at_home: double (nullable = true)
 |-- pop_not_stay_at_home: double (nullable = true)
 |-- trips: double (nullable = true)
 |-- trips_1: double (nullable = true)
 |-- trips_1_3: double (nullable = true)
 |-- trips_3_5: double (nullable = true)
 |-- trips_5_10: double (nullable = true)
 |-- trips_10_25: double (nullable = true)
 |-- trips_25_50: double (nullable = true)
 |-- trips_50_100: double (nullable = true)
 |-- trips_100_250: double (nullable = true)
 |-- trips_250_500: double (nullable = true)
 |-- trips_500: double (nullable = true)
 |-- row_id: string (nullable = true)
 |-- week: integer (nullable = true)
 |-- month: integer (nullable = true)



In [29]:
df.columns

['level',
 'date',
 'state_fips',
 'state_code',
 'county_fips',
 'county',
 'pop_stay_at_home',
 'pop_not_stay_at_home',
 'trips',
 'trips_1',
 'trips_1_3',
 'trips_3_5',
 'trips_5_10',
 'trips_10_25',
 'trips_25_50',
 'trips_50_100',
 'trips_100_250',
 'trips_250_500',
 'trips_500',
 'row_id',
 'week',
 'month']

In [30]:
df.show(5, truncate=False)

+--------+-------------------+----------+----------+-----------+------+----------------+--------------------+-------------+------------+------------+------------+------------+------------+-----------+------------+-------------+-------------+---------+-----------------+----+-----+
|level   |date               |state_fips|state_code|county_fips|county|pop_stay_at_home|pop_not_stay_at_home|trips        |trips_1     |trips_1_3   |trips_3_5   |trips_5_10  |trips_10_25 |trips_25_50|trips_50_100|trips_100_250|trips_250_500|trips_500|row_id           |week|month|
+--------+-------------------+----------+----------+-----------+------+----------------+--------------------+-------------+------------+------------+------------+------------+------------+-----------+------------+-------------+-------------+---------+-----------------+----+-----+
|National|2019-01-01 00:00:00|NULL      |NULL      |NULL       |NULL  |7.7433867E7     |2.48733553E8        |8.97784368E8 |2.41667151E8|2.34284795E8|1.080789

In [32]:
from pyspark.sql.functions import col, count, when
from pyspark.sql.types import DoubleType, IntegerType, LongType, FloatType

numeric_types = (DoubleType, IntegerType, LongType, FloatType)

null_counts = df.select([
    count(
        when(
            col(c).isNull() | 
            (col(c).cast("double").isNotNull() & col(c).isNaN()),
            c
        )
    ).alias(c)
    if isinstance(df.schema[c].dataType, numeric_types)
    else count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

null_counts.show(truncate=False)

[Stage 13:>                                                       (0 + 16) / 16]

+-----+----+----------+----------+-----------+------+----------------+--------------------+-----+-------+---------+---------+----------+-----------+-----------+------------+-------------+-------------+---------+------+----+-----+
|level|date|state_fips|state_code|county_fips|county|pop_stay_at_home|pop_not_stay_at_home|trips|trips_1|trips_1_3|trips_3_5|trips_5_10|trips_10_25|trips_25_50|trips_50_100|trips_100_250|trips_250_500|trips_500|row_id|week|month|
+-----+----+----------+----------+-----------+------+----------------+--------------------+-----+-------+---------+---------+----------+-----------+-----------+------------+-------------+-------------+---------+------+----+-----+
|0    |0   |1552      |1552      |80704      |80704 |41295           |41295               |41295|41295  |41295    |41295    |41295     |41295      |41295      |41295       |41295        |41295        |41295    |0     |0   |0    |
+-----+----+----------+----------+-----------+------+----------------+----------

In [33]:
df_national = df.filter(df.level == "National")
df_state = df.filter(df.level == "State")
df_county = df.filter(df.level == "County")

In [34]:
print("National:", df_national.count())
print("State:", df_state.count())
print("County:", df_county.count())

National: 1552


State: 79152


[Stage 22:>                                                       (0 + 16) / 16]

County: 4919296


In [35]:
from pyspark.sql.functions import col

quantiles = df.approxQuantile("trips", [0.25, 0.75], 0)

Q1, Q3 = quantiles
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df.filter((col("trips") < lower) | (col("trips") > upper))

print("Trips Outliers:", outliers.count())

[Stage 28:>                                                       (0 + 16) / 16]

Trips Outliers: 677760


In [36]:
df_clean = df.filter(
    (col("trips") >= lower) & 
    (col("trips") <= upper)
)

print("Clean Rows:", df_clean.count())

[Stage 31:==========>                                             (3 + 13) / 16]

Clean Rows: 4280945


In [37]:
df_clean.filter(col("trips") < 0).count()

0

In [38]:
df_clean.count() - df_clean.dropDuplicates().count()

0

In [39]:
df_clean.write \
    .mode("overwrite") \
    .partitionBy("level", "month") \
    .parquet("data/processed/mobility_clean")

26/03/02 20:17:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/02 20:17:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/02 20:17:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/02 20:17:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/02 20:17:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/02 20:17:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/02 20:17:10 WARN MemoryManager: Total allocation exceeds 95.

[4090.700s][warning][gc,alloc] Executor task launch worker for task 7.0 in stage 46.0 (TID 293): Retried waiting for GCLocker too often allocating 13052 words


26/03/02 20:17:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 50.67% for 15 writers
26/03/02 20:17:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 54.29% for 14 writers
26/03/02 20:17:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 50.67% for 15 writers
26/03/02 20:17:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 47.50% for 16 writers
26/03/02 20:17:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 50.67% for 15 writers
26/03/02 20:17:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 54.29% for 14 writers
26/03/02 20:17:12 WARN MemoryManager: Total allocation exceeds 9

In [40]:
!ls data/processed/

mobility_clean


In [42]:
df_clean = df_clean.repartition(50)

In [43]:
df_clean.write \
    .mode("overwrite") \
    .partitionBy("level", "month") \
    .parquet("data/processed/mobility_clean")

26/03/02 20:18:34 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/02 20:18:34 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/02 20:18:34 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/02 20:18:34 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/02 20:18:34 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/02 20:18:34 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/02 20:18:34 WARN MemoryManager: Total allocation exceeds 95.